export CAC_API_KEY="API KEY GOES HERE" # Run this in terminal or Use .env file and add it to .gitignore

In [4]:
import pandas as pd
import requests
import os
# Environment variable for API key
# api_key = os.getenv("CAC_API_KEY")

# .env file
from dotenv import load_dotenv
load_dotenv()  # loads .env
api_key = os.getenv("CAC_API_KEY")

host = "http://a-colon.rz.uni-mannheim.de"
model = "mistral-nemo"


In [30]:
df = pd.read_csv("dataset/all_multi_paragraphs_2022_2026_translated.csv")

test_df = df.tail(10)
test_df

,article_id,pub,par_index,text_block,group,length,text_block_en
24390,LVZ-doc7j5h23b2qow1k8d6jmbc,LVZ,6,Erst da wurde die Bergier-Historikerkommission...,JUDE,208,Only now was the Bergier-Historikerkommission ...
24391,LVZ-doc7j7jed03saoarzghh4s,LVZ,19,"Zarka lacht über diesen Vergleich, findet die ...",JUDE,142,Zarka laughs at this comparison and doesn't th...
24392,LVZ-doc7j7jxw522agl007nkj9,LVZ,2,Schiedsrichter Felix Zwayer pfeift nach dem Wi...,JUDE,225,Referee Felix Zwayer is currently not officiat...
24393,LVZ-doc7j7jsnraywn154o1afa0,LVZ,2,"Marco Rose sah ein bisschen so aus wie einer, ...",JUDE,331,Marco Rose looked a bit like someone who had j...
24394,LVZ-doc7j9jvujsrvckqeps12z,LVZ,7,Fritzsche hingegen warnt vor Geschichtsklitter...,JUDE,150,"Fritzsche, on the other hand, warns against hi..."
24395,LVZ-doc7ja4inwy8au12alaeh57,LVZ,3,"Das Projekt, die Ausstellung "" In Rosas Schatt...",JUDE,218,"The project to bring the exhibition ""In Rosas ..."
24396,LVZ-doc7jbqfuefxqxv9q3vge6,LVZ,1,\n\nEs ist eines der großen Rätsel der Geschic...,JUDE,239,It is one of the great mysteries of history: W...
24397,LVZ-doc7jbqfuefxqxv9q3vge6,LVZ,2,Es ist eines der großen Rätsel der Geschichte:...,JUDE,313,One of the great enigmas of history is: Who be...
24398,LVZ-doc7jbq5r8asxk9xtca2dy,LVZ,1,\n\nEs ist eines der großen Rätsel der Geschic...,JUDE,239,It is one of the great puzzles of history: Who...
24399,LVZ-doc7jbq5r8asxk9xtca2dy,LVZ,2,Es ist eines der großen Rätsel der Geschichte:...,JUDE,313,One of the great enigmas in history is: who be...


### API TEST

In [ ]:
# Few-shot examples
example_instructions_expl = "Please classify each paragraph according to the given labels and explain why."
example_fewshot = {
    "input": [
        "This is the first example paragraph.",
        "This is the second example paragraph."
    ],
    "explanation": [
        "This paragraph talks about topic A, so label is X.",
        "This paragraph mentions topic B, so label is Y."
    ],
    "label": ["X", "Y"]
}
example_reminder_expl = "Always provide the reasoning before the final label."

# Function to construct few-shot prompt for each paragraph
def generate_prompt(paragraph):
    prompt = "Instructions: " + example_instructions_expl + "\n\n"
    for inp, expl, label in zip(example_fewshot["input"], example_fewshot["explanation"], example_fewshot["label"]):
        prompt += f"Example:\nInput: {inp}\nExplanation: {expl}\nLabel: {label}\n\n"
    prompt += example_reminder_expl + "\n\n"
    prompt += "Now classify or explain this paragraph:\n" + paragraph
    return prompt

# Loop through the dataset and call API
results = []

for idx, row in test_df.iterrows():
    paragraph = row['text_block']
    article_id = row['article_id']
    
    prompt = generate_prompt(paragraph)
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt}
        ]
    }
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    url = f"{host}/v1/chat/completions"
    response = requests.post(url, json=payload, headers=headers)
    
    if response.status_code == 200:
        content = response.json()['choices'][0]['message']['content']
        results.append({
            "article_id": article_id,
            "par_index": row['par_index'],
            "text_block": paragraph,
            "model_output": content
        })
    else:
        print(f"Error with article {article_id}, paragraph {row['par_index']}. Status code: {response.status_code}")
        results.append({
            "article_id": article_id,
            "par_index": row['par_index'],
            "text_block": paragraph,
            "model_output": None
        })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
import pandas as pd
import requests
import os

pd.head -3 dataset/all_multi_paragraphs_2022_2026.csv